# Normalized SPECTER Stage 3-4 Rerun

Rerun of the downstream Stage 3/4 experiment using empty-string-centered, unit-normalized SPECTER2 embeddings. The legacy unnormalized-cache outputs from notebook 6 remain preserved as the baseline.

This notebook writes only normalized-SPECTER outputs, reruns the six Stage 3 branches, and can launch the corrected normalized Stage 4 generation branches.


In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())

# Match the working Colab setup used by Notebook 2:
# code repo in /content/neurovlm_gnn, Drive used for data and run outputs.
REPO_URL = os.environ.get("NEUROVLM_REPO_URL", "https://github.com/neurovlm/neurovlm.git")
REPO_BRANCH = os.environ.get("NEUROVLM_REPO_BRANCH", "neurovlm_gnn")
REPO_DIR = Path(os.environ.get("NEUROVLM_REPO_DIR", "/content/neurovlm_gnn"))
DRIVE_ROOT = Path(os.environ.get("NEUROVLM_DRIVE_ROOT", "/content/drive/MyDrive/neurovlm"))
INSTALL_DEPENDENCIES = os.environ.get("NEUROVLM_INSTALL_DEPS", "1") == "1"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def run_cmd(cmd, cwd=None, *, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        if result.stderr.strip():
            print(result.stderr.strip())
        if check:
            raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, cmd))}")
    return result

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run_cmd(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a git checkout. Set NEUROVLM_REPO_DIR to a clean path "
            "or remove that folder, then rerun this cell."
        )
    run_cmd(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
    checkout = run_cmd(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=False)
    if checkout.returncode != 0:
        run_cmd(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"])
    run_cmd(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])

os.chdir(REPO_DIR)

if INSTALL_DEPENDENCIES:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "nilearn", "nibabel", "huggingface-hub", "safetensors", "adapters", "transformers", "pyarrow", "matplotlib", "pandas", "scikit-learn", "tqdm", "umap-learn"])
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-e", ".[viz,notebook,metrics]"])

sys.path.insert(0, str(REPO_DIR / "experiments" / "3dcnn"))
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR))

print("Working directory:", os.getcwd())
print("Repo branch:", run_cmd(["git", "branch", "--show-current"], cwd=REPO_DIR, check=False).stdout.strip())
print("Drive root:", DRIVE_ROOT)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

from atlas_free_cnn.pipeline_outputs import (
    create_stage2_stage3_stage4_run_dir,
    git_info,
    write_json,
    write_status_report,
    write_readme_what_to_look_at,
    write_table,
    flatten_stage3_metrics,
    ae_source_metric_columns,
)
from atlas_free_cnn.stage1_selection_integration import IntegrationConfig, integrate_completed_stage1_selection
from atlas_free_cnn.notebook_utils import (
    DOMAIN_DIRS,
    SPECIALIZED_BRANCHES,
    CORRECTED_STAGE4_CHECKPOINT,
    CORRECTED_STAGE4_DIRNAME,
    NORMALIZED_STAGE3_CHECKPOINT,
    NORMALIZED_STAGE3_DIRNAME,
    discover_unified_split_dir as shared_discover_unified_split_dir,
    build_normalized_specter2_cache,
    download_text_embedding_cache,
    hf_download_first_available as shared_hf_download_first_available,
    print_relevant_text_embedding_hf_files,
    print_split_files,
    required_text_records_from_jsonls,
    resolve_text_embedding_cache,
    sha256_file,
    split_file_fingerprints,
    text_embedding_metadata_fields,
    validate_text_embedding_cache,
)


## Config Switches

Top-level controls for building/downloading the normalized cache, rerunning the six normalized Stage 3 branches, and running the six corrected normalized Stage 4 branches.


In [ ]:
# 6a top-level controls
CREATE_OR_DOWNLOAD_NORMALIZED_CACHE = True
RERUN_STAGE3_NORMALIZED = True
RUN_CORRECTED_STAGE4_NORMALIZED = True
RUN_FRESH_STAGE3_PROJECTOR_CONTROL = False
UPLOAD_RESULTS_TO_DRIVE = True

TEXT_EMBEDDING_CONVENTION = os.environ.get("NEUROVLM_TEXT_EMBEDDING_CONVENTION", "normalized_specter2")


In [ ]:
HF_DATASET_REPO = os.environ.get("NEUROVLM_ATLAS_FREE_HF_REPO", "neurovlm/atlas_free_cnn_dataset")
LOCAL_UNIFIED_CACHE_DIR = REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl_rebuild"
LOCAL_SPLIT_DIR = LOCAL_UNIFIED_CACHE_DIR / "splits"
LOCAL_PACK_DIR = REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/hf_atlas_free_cnn_rebuild"
LOCAL_TEXT_CACHE_DIR = REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/text_embeddings"
TEXT_EMBEDDING_SPEC = resolve_text_embedding_cache(
    TEXT_EMBEDDING_CONVENTION,
    repo_dir=REPO_DIR,
    local_cache_dir=LOCAL_TEXT_CACHE_DIR,
    hf_repo=HF_DATASET_REPO,
)
if TEXT_EMBEDDING_SPEC["convention"] != "normalized_specter2":
    raise ValueError("Notebook 6a writes normalized Stage 3/4 outputs; set TEXT_EMBEDDING_CONVENTION='normalized_specter2'.")
TEXT_EMBEDDING_PREPROCESSING = TEXT_EMBEDDING_SPEC["preprocessing"]


def hf_download_first_available(filenames, local_dir: Path) -> Path:
    return shared_hf_download_first_available(
        list(filenames),
        local_dir,
        dataset_repo=HF_DATASET_REPO,
    )


def discover_unified_split_dir() -> Path:
    return shared_discover_unified_split_dir(
        repo_dir=REPO_DIR,
        drive_root=DRIVE_ROOT,
        dataset_repo=HF_DATASET_REPO,
        local_unified_cache_dir=LOCAL_UNIFIED_CACHE_DIR,
        local_split_dir=LOCAL_SPLIT_DIR,
        local_pack_dir=LOCAL_PACK_DIR,
    )

UNIFIED_SPLIT_DIR = discover_unified_split_dir()
TRAIN_JSONL = str(UNIFIED_SPLIT_DIR / "train.jsonl")
VAL_JSONL = str(UNIFIED_SPLIT_DIR / "val.jsonl")
TEST_JSONL = str(UNIFIED_SPLIT_DIR / "test.jsonl")
SPLIT_FINGERPRINTS = split_file_fingerprints(UNIFIED_SPLIT_DIR)
print("Unified split dir:", UNIFIED_SPLIT_DIR)
print("Train JSONL:", TRAIN_JSONL)

RUN_MODE = "downstream_only"
DATA_MODE = "mixed"
RERUN_STAGE1_CHECKPOINT_EVALUATION = False

# Required: explicit selected checkpoint paths. By default this uses the completed
# Stage 1A pretraining run for the mixed baseline and the completed Stage 1B
# fine-tuning run for PubMed/Nilearn/NeuroVault. Override individual checkpoint
# paths with NEUROVLM_*_AE_CKPT if your Drive layout differs.
AE_PRETRAINED_RUN_ROOT_VALUE = os.environ.get("NEUROVLM_AE_PRETRAINED_RUN_ROOT", "").strip()
AE_FINETUNED_RUN_ROOT_VALUE = os.environ.get("NEUROVLM_AE_FINETUNED_RUN_ROOT", "").strip()
AE_PRETRAINED_RUN_ROOT = Path(AE_PRETRAINED_RUN_ROOT_VALUE).expanduser() if AE_PRETRAINED_RUN_ROOT_VALUE else DRIVE_ROOT / "runs_atlas_free_cnn_ae_ablation/ae_ablation_20260623_165729"
AE_FINETUNED_RUN_ROOT = Path(AE_FINETUNED_RUN_ROOT_VALUE).expanduser() if AE_FINETUNED_RUN_ROOT_VALUE else DRIVE_ROOT / "runs_atlas_free_cnn_ae_ablation/ae_ablation_20260624_190738"

CONFIGURED_SELECTED_AE_CHECKPOINTS = {
    "mixed_stage1a": {
        "path": os.environ.get(
            "NEUROVLM_MIXED_STAGE1A_AE_CKPT",
            str(AE_PRETRAINED_RUN_ROOT / "01_stage1_ae_pretraining/mixed_baseline_raw_mse/checkpoints/best_top1_dice.pt"),
        ),
        "stage": "stage1a",
        "training_domain": "mixed",
        "checkpoint_name": "best_top1_dice.pt",
        "selection_reason": "locked_downstream_stage1_selection",
        "evaluation_status": "completed",
    },
    "mixed_to_pubmed_stage1b": {
        "path": os.environ.get(
            "NEUROVLM_PUBMED_STAGE1B_AE_CKPT",
            str(AE_FINETUNED_RUN_ROOT / "02_stage1b_ae_finetuning/pubmed/checkpoints/best_top1_dice.pt"),
        ),
        "stage": "stage1b",
        "training_domain": "pubmed",
        "checkpoint_name": "best_top1_dice.pt",
        "selection_reason": "locked_downstream_stage1_selection",
        "evaluation_status": "completed",
    },
    "mixed_to_nilearn_stage1b": {
        "path": os.environ.get(
            "NEUROVLM_NILEARN_STAGE1B_AE_CKPT",
            str(AE_FINETUNED_RUN_ROOT / "02_stage1b_ae_finetuning/nilearn/checkpoints/best_val_loss.pt"),
        ),
        "stage": "stage1b",
        "training_domain": "nilearn",
        "checkpoint_name": "best_val_loss.pt",
        "selection_reason": "locked_downstream_stage1_selection",
        "evaluation_status": "completed",
    },
    "mixed_to_neurovault_stage1b": {
        "path": os.environ.get(
            "NEUROVLM_NEUROVAULT_STAGE1B_AE_CKPT",
            str(AE_FINETUNED_RUN_ROOT / "02_stage1b_ae_finetuning/neurovault/checkpoints/best_top5_dice.pt"),
        ),
        "stage": "stage1b",
        "training_domain": "neurovault",
        "checkpoint_name": "best_top5_dice.pt",
        "selection_reason": "locked_downstream_stage1_selection",
        "evaluation_status": "completed",
    },
}

# Optional provenance-only notebook-7 evaluation output folders. Leave blank if unavailable.
STAGE1A_EVALUATION_DIR_VALUE = os.environ.get("NEUROVLM_STAGE1A_EVALUATION_DIR", "").strip()
STAGE1B_EVALUATION_DIR_VALUE = os.environ.get("NEUROVLM_STAGE1B_EVALUATION_DIR", "").strip()
STAGE1A_EVALUATION_DIR = Path(STAGE1A_EVALUATION_DIR_VALUE).expanduser() if STAGE1A_EVALUATION_DIR_VALUE else None
STAGE1B_EVALUATION_DIR = Path(STAGE1B_EVALUATION_DIR_VALUE).expanduser() if STAGE1B_EVALUATION_DIR_VALUE else None

RUN_STAGE1_SELECTION_INTEGRATION = True
RERUN_STAGE3 = os.environ.get("NEUROVLM_RERUN_STAGE3_NORMALIZED", "1" if RERUN_STAGE3_NORMALIZED else "0") == "1"
RUN_STAGE3_CONTRASTIVE = RERUN_STAGE3
RUN_STAGE4_TEXT_TO_BRAIN = os.environ.get("NEUROVLM_RUN_CORRECTED_STAGE4_NORMALIZED", "1" if RUN_CORRECTED_STAGE4_NORMALIZED else "0") == "1"
RUN_STAGE5_GENERATION_EVAL = True
COMPLETED_STAGE3_RUN_ROOT_VALUE = os.environ.get("NEUROVLM_COMPLETED_STAGE3_RUN_ROOT", "").strip()
DEFAULT_COMPLETED_STAGE3_RUN_ROOT = DRIVE_ROOT / "runs_stage2_stage3_stage4/stage2_stage3_stage4_20260625_210801"
COMPLETED_STAGE3_RUN_ROOT = Path(COMPLETED_STAGE3_RUN_ROOT_VALUE).expanduser() if COMPLETED_STAGE3_RUN_ROOT_VALUE else DEFAULT_COMPLETED_STAGE3_RUN_ROOT
STAGE4_TEXT_EMBEDDING_CACHE_VALUE = os.environ.get("NEUROVLM_STAGE4_TEXT_EMBEDDING_CACHE", "").strip()
STAGE4_TEXT_EMBEDDING_CACHE = Path(STAGE4_TEXT_EMBEDDING_CACHE_VALUE).expanduser() if STAGE4_TEXT_EMBEDDING_CACHE_VALUE else None
STAGE4_TEXT_EMBEDDING_CACHE_FILENAME = os.environ.get("NEUROVLM_STAGE4_TEXT_EMBEDDING_CACHE_FILENAME", "").strip()

domain_dirs = DOMAIN_DIRS
specialized_dirs = SPECIALIZED_BRANCHES
NUM_WORKERS = int(os.environ.get("NEUROVLM_NUM_WORKERS", "4" if IN_COLAB else "0"))
EVAL_NUM_WORKERS = int(os.environ.get("NEUROVLM_EVAL_NUM_WORKERS", str(NUM_WORKERS)))
PREFETCH_FACTOR = int(os.environ.get("NEUROVLM_PREFETCH_FACTOR", "4"))
METRICS_DEVICE = os.environ.get("NEUROVLM_METRICS_DEVICE", "cuda")

BASE_OUTPUT_DIR = DRIVE_ROOT if "DRIVE_ROOT" in globals() and UPLOAD_RESULTS_TO_DRIVE else Path(".")
STAGE1_SELECTION_INTEGRATION_OUTPUT_ROOT = DRIVE_ROOT / "runs_stage1_selection_integration_6a_normalized_specter" if "DRIVE_ROOT" in globals() else Path("runs_stage1_selection_integration_6a_normalized_specter")


In [ ]:
paths = create_stage2_stage3_stage4_run_dir(BASE_OUTPUT_DIR, prefix="6a_results", branch_stage_dirs=())
RUN_DIR = Path(paths["run_dir"])
print(f"Run directory: {RUN_DIR}")

metadata_dir = Path(paths["metadata"])
metadata_dir.mkdir(parents=True, exist_ok=True)
write_json(metadata_dir / "run_config.json", {
    "RUN_MODE": RUN_MODE,
    "DATA_MODE": DATA_MODE,
    "AE_PRETRAINED_RUN_ROOT": str(AE_PRETRAINED_RUN_ROOT),
    "AE_FINETUNED_RUN_ROOT": str(AE_FINETUNED_RUN_ROOT),
    "CONFIGURED_SELECTED_AE_CHECKPOINTS": CONFIGURED_SELECTED_AE_CHECKPOINTS,
    "STAGE1A_EVALUATION_DIR": str(STAGE1A_EVALUATION_DIR or ""),
    "STAGE1B_EVALUATION_DIR": str(STAGE1B_EVALUATION_DIR or ""),
    "RERUN_STAGE1_CHECKPOINT_EVALUATION": RERUN_STAGE1_CHECKPOINT_EVALUATION,
    "RUN_STAGE1_SELECTION_INTEGRATION": RUN_STAGE1_SELECTION_INTEGRATION,
    "RERUN_STAGE3": RERUN_STAGE3,
    "RUN_STAGE3_CONTRASTIVE": RUN_STAGE3_CONTRASTIVE,
    "COMPLETED_STAGE3_RUN_ROOT": str(COMPLETED_STAGE3_RUN_ROOT or ""),
    "RUN_STAGE4_TEXT_TO_BRAIN": RUN_STAGE4_TEXT_TO_BRAIN,
    "STAGE4_TEXT_EMBEDDING_CACHE": str(STAGE4_TEXT_EMBEDDING_CACHE or ""),
    "STAGE4_TEXT_EMBEDDING_CACHE_FILENAME": STAGE4_TEXT_EMBEDDING_CACHE_FILENAME,
    "TEXT_EMBEDDING_CONVENTION": TEXT_EMBEDDING_CONVENTION,
    "TEXT_EMBEDDING_SPEC": TEXT_EMBEDDING_SPEC,
    **text_embedding_metadata_fields(TEXT_EMBEDDING_SPEC),
    "RUN_STAGE5_GENERATION_EVAL": RUN_STAGE5_GENERATION_EVAL,
})
write_json(metadata_dir / "git_info.json", git_info(REPO_DIR))
write_json(metadata_dir / "unified_split_fingerprints.json", SPLIT_FINGERPRINTS)
(metadata_dir / "environment.txt").write_text(sys.version)


## Stage 1 Selection Integration: Validate Explicit Checkpoint Paths

In [ ]:
SELECTED_AE_CHECKPOINTS = {}
STAGE2_STAGE3_STAGE4_INPUT_MANIFEST = None
STAGE1_SELECTION_INTEGRATION_DIR = None
stage1_selection_status = "not_requested"

if RUN_STAGE1_SELECTION_INTEGRATION:
    integration_result = integrate_completed_stage1_selection(
        IntegrationConfig(
            output_root=STAGE1_SELECTION_INTEGRATION_OUTPUT_ROOT,
            selected_checkpoints=CONFIGURED_SELECTED_AE_CHECKPOINTS,
            stage1a_evaluation_dir=STAGE1A_EVALUATION_DIR,
            stage1b_evaluation_dir=STAGE1B_EVALUATION_DIR,
            rerun_stage1_checkpoint_evaluation=RERUN_STAGE1_CHECKPOINT_EVALUATION,
        )
    )
    stage1_selection_status = integration_result["status"]
    STAGE1_SELECTION_INTEGRATION_DIR = Path(integration_result["output_dir"])
    STAGE2_STAGE3_STAGE4_INPUT_MANIFEST = STAGE1_SELECTION_INTEGRATION_DIR / "03_downstream_usage/stage2_stage3_stage4_input_manifest.json"
    downstream_manifest = json.loads(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST.read_text())
    SELECTED_AE_CHECKPOINTS = downstream_manifest["selected_ae_checkpoints"]
    write_json(metadata_dir / "stage1_selection_integration_result.json", integration_result)
    write_json(metadata_dir / "stage2_stage3_stage4_input_manifest_pointer.json", {"path": str(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST)})
    validation_report = STAGE1_SELECTION_INTEGRATION_DIR / "02_selected_checkpoint_registry/selected_checkpoint_validation.json"
    if stage1_selection_status not in {"completed", "completed_with_warnings"}:
        print("Stage 1 checkpoint validation failed:", stage1_selection_status)
        print("Validation report:", validation_report)
        for row in integration_result.get("blocking_checkpoints", []):
            print(f"- {row.get('key')}: {row.get('status')} -> {row.get('checkpoint_path')}")
            if row.get("warnings"):
                print("  warnings:", row.get("warnings"))
        raise RuntimeError(
            f"Stage 1 selection integration failed: {stage1_selection_status}. "
            f"Set NEUROVLM_AE_CHECKPOINT_ROOT or the individual NEUROVLM_*_AE_CKPT paths. "
            f"See {validation_report}"
        )
    print("Stage 1 checkpoint validation:", stage1_selection_status)
    print("Integration output:", STAGE1_SELECTION_INTEGRATION_DIR)
    print("Downstream manifest:", STAGE2_STAGE3_STAGE4_INPUT_MANIFEST)
else:
    print("Stage 1 selection integration not requested")

## Preserved Stage 1 Evaluation Tables

In [ ]:
if STAGE1_SELECTION_INTEGRATION_DIR:
    for path in [
        STAGE1_SELECTION_INTEGRATION_DIR / "01_existing_evaluation_tables/mixed_stage1a_checkpoint_selection.csv",
        STAGE1_SELECTION_INTEGRATION_DIR / "01_existing_evaluation_tables/pubmed_stage1b_checkpoint_selection.csv",
        STAGE1_SELECTION_INTEGRATION_DIR / "01_existing_evaluation_tables/nilearn_stage1b_checkpoint_selection.csv",
        STAGE1_SELECTION_INTEGRATION_DIR / "01_existing_evaluation_tables/neurovault_stage1b_checkpoint_selection.csv",
        STAGE1_SELECTION_INTEGRATION_DIR / "02_selected_checkpoint_registry/selected_ae_checkpoints_for_stage2_stage3_stage4.json",
        STAGE1_SELECTION_INTEGRATION_DIR / "02_selected_checkpoint_registry/selected_checkpoint_validation.json",
        STAGE1_SELECTION_INTEGRATION_DIR / "03_downstream_usage/six_run_ae_assignment.csv",
    ]:
        print(path, "exists=", path.exists())
else:
    print("No Stage 1 selection integration output available")

## Part 1: Build or Download Normalized SPECTER2 Cache

This cell lists the Hugging Face repository, builds or downloads `text_embeddings/specter2_stage3_stage4_emptycentered_unitnorm.pt`, validates 768-dimensional unit norms, and fails before training if the wrong cache is loaded.


In [ ]:
def required_primary_text_records() -> dict:
    return required_text_records_from_jsonls([TRAIN_JSONL, VAL_JSONL, TEST_JSONL])


def validate_resolved_text_cache(spec: dict) -> dict:
    requirements = required_primary_text_records()
    audit = validate_text_embedding_cache(
        spec,
        required_text_ids=requirements["text_ids"],
        required_texts=requirements["texts"],
        expected_dim=spec["expected_dim"],
        expect_unit_norm=spec["expect_unit_norm"],
        mean_norm_tol=1e-3,
    )
    print(f"{spec['convention']} text embedding cache stats")
    print(json.dumps(audit["stats"], indent=2))
    return audit


if CREATE_OR_DOWNLOAD_NORMALIZED_CACHE:
    print_relevant_text_embedding_hf_files(HF_DATASET_REPO)

TEXT_EMBEDDING_SPEC = resolve_text_embedding_cache(
    TEXT_EMBEDDING_CONVENTION,
    repo_dir=REPO_DIR,
    local_cache_dir=LOCAL_TEXT_CACHE_DIR,
    hf_repo=HF_DATASET_REPO,
)
TEXT_EMBEDDING_CACHE = Path(TEXT_EMBEDDING_SPEC["local_cache_path"]).expanduser()
if not TEXT_EMBEDDING_CACHE.exists():
    downloaded = False
    try:
        TEXT_EMBEDDING_CACHE = download_text_embedding_cache(TEXT_EMBEDDING_SPEC)
        downloaded = True
    except Exception as exc:
        print(f"{TEXT_EMBEDDING_SPEC['convention']} cache was not available on HF yet:", exc)
    if not downloaded and CREATE_OR_DOWNLOAD_NORMALIZED_CACHE:
        TEXT_EMBEDDING_CACHE = build_normalized_specter2_cache(
            TEXT_EMBEDDING_SPEC,
            repo_dir=REPO_DIR,
            python_executable=sys.executable,
        )

if not TEXT_EMBEDDING_CACHE.exists():
    raise FileNotFoundError(f"Text embedding cache is missing: {TEXT_EMBEDDING_CACHE}")

TEXT_EMBEDDING_SPEC["local_cache_path"] = str(TEXT_EMBEDDING_CACHE)
TEXT_EMBEDDING_AUDIT = validate_resolved_text_cache(TEXT_EMBEDDING_SPEC)
TEXT_EMBEDDING_METADATA = text_embedding_metadata_fields(TEXT_EMBEDDING_SPEC, TEXT_EMBEDDING_AUDIT)
write_json(metadata_dir / f"text_cache_audit_{TEXT_EMBEDDING_CONVENTION}.json", {"spec": TEXT_EMBEDDING_SPEC, "audit": TEXT_EMBEDDING_AUDIT})
write_json(metadata_dir / f"text_cache_metadata_{TEXT_EMBEDDING_CONVENTION}.json", TEXT_EMBEDDING_METADATA)

NORMALIZED_PATHS = {
    "pt": Path(TEXT_EMBEDDING_SPEC["local_cache_path"]),
    "metadata": Path(TEXT_EMBEDDING_SPEC["metadata_local_path"]),
    "validation": Path(TEXT_EMBEDDING_SPEC["validation_local_path"]),
    "index": Path(TEXT_EMBEDDING_SPEC["index_local_path"]),
}
NORMALIZED_SPECTER_CACHE = TEXT_EMBEDDING_CACHE
NORMALIZED_CACHE_AUDIT = TEXT_EMBEDDING_AUDIT
TEXT_EMBEDDING_CACHE = NORMALIZED_SPECTER_CACHE
STAGE4_TEXT_EMBEDDING_CACHE = NORMALIZED_SPECTER_CACHE
print("TEXT_EMBEDDING_CONVENTION =", TEXT_EMBEDDING_CONVENTION)
print("TEXT_EMBEDDING_CACHE =", TEXT_EMBEDDING_CACHE)
print("TEXT_EMBEDDING_PREPROCESSING =", TEXT_EMBEDDING_PREPROCESSING)


## Stage 2/3: Six Controlled Encoder Initialization Runs

In [ ]:
TEXT_EMBEDDING_CACHE = NORMALIZED_SPECTER_CACHE
print("Normalized Stage 3 text embedding cache:", TEXT_EMBEDDING_CACHE)
print("Text preprocessing:", TEXT_EMBEDDING_PREPROCESSING)

def copy_stage3_checkpoint_aliases(ckpt_dir: Path) -> None:
    best = ckpt_dir / "best_ale_cnn.pt"
    last = ckpt_dir / "last_ale_cnn.pt"
    if best.exists():
        shutil.copy2(best, ckpt_dir / "best_val_normalized_recall_auc.pt")
    if last.exists():
        shutil.copy2(last, ckpt_dir / "last.pt")

if RUN_STAGE3_CONTRASTIVE:
    if not STAGE2_STAGE3_STAGE4_INPUT_MANIFEST or not Path(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST).exists():
        raise RuntimeError("Stage 2/3 requires the validated Stage 1 selected-checkpoint manifest")
    downstream_manifest = json.loads(Path(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST).read_text())
    domain_dirs = {"pubmed": "01_pubmed", "nilearn": "02_nilearn", "neurovault": "03_neurovault"}
    specialized_dirs = {
        "pubmed": "specialized_mixed_to_pubmed",
        "nilearn": "specialized_mixed_to_nilearn",
        "neurovault": "specialized_mixed_to_neurovault",
    }
    for run in downstream_manifest["six_stage2_stage3_stage4_runs"]:
        domain = run["domain"]
        branch = "baseline_mixed_stage1a" if run["type"] == "baseline" else specialized_dirs[domain]
        stage3_dir = RUN_DIR / domain_dirs[domain] / branch / "stage3_normalized_specter"
        ckpt_dir = stage3_dir / "checkpoints"
        complete_marker = stage3_dir / "NORMALIZED_STAGE3_COMPLETE.json"
        if complete_marker.exists():
            print("Skipping complete normalized Stage 3 run:", stage3_dir)
            copy_stage3_checkpoint_aliases(ckpt_dir)
            continue
        ae_entry = downstream_manifest["selected_ae_checkpoints"][run["ae_registry_key"]]
        checkpoint_name = Path(ae_entry["checkpoint_name"]).stem
        cmd = [
            sys.executable, "experiments/3dcnn/atlas_free_cnn/training/train_ale_cnn.py",
            "--mode", "atlas_free",
            "--model", "ale_3dcnn",
            "--train-jsonl", TRAIN_JSONL,
            "--val-jsonl", VAL_JSONL,
            "--test-jsonl", TEST_JSONL,
            "--text-embedding-cache", str(TEXT_EMBEDDING_CACHE),
            "--domain", domain,
            "--target-shape", "36,45,38",
            "--epochs", "150",
            "--early-stopping-patience", "25",
            "--val-interval", "1",
            "--batch-size", "512",
            "--base-channels", "64",
            "--num-blocks", "4",
            "--out-dim", "384",
            "--lr-cnn", "0.0001",
            "--lr-proj", "0.00001",
            "--weight-decay", "0.0001",
            "--warmup-epochs", "5",
            "--temperature", "0.07",
            "--dropout", "0.1",
            "--norm", "group",
            "--pooling", "max",
            "--encoder-init", "autoencoder_pretrained",
            "--ae-ckpt-path", str(ae_entry["path"]),
            "--ae-init-variant", str(run["ae_registry_key"]),
            "--ae-checkpoint-selection", checkpoint_name,
            "--text-proj-init", "pretrained_infonce",
            "--monitor-metric", "paper_recall_curve_auc",
            "--run-dir", str(stage3_dir),
            "--checkpoint-dir", str(ckpt_dir),
        ]
        print("Launching normalized Stage 3", f"{run['run']}_normalized_specter")
        print(" ".join(cmd))
        _env3 = {**os.environ, "PYTHONUNBUFFERED": "1"}
        _proc3 = subprocess.Popen(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1, env=_env3)
        for _line in _proc3.stdout:
            print(_line, end='', flush=True)
        _proc3.wait()
        if _proc3.returncode != 0:
            raise RuntimeError(f"Normalized Stage 3 run failed for {run['run']} with exit code {_proc3.returncode}")
        copy_stage3_checkpoint_aliases(ckpt_dir)
        write_json(stage3_dir / "normalized_specter_provenance.json", {
            "run": f"{run['run']}_normalized_specter",
            "normalized_cache": str(TEXT_EMBEDDING_CACHE),
            "normalized_cache_sha256": sha256_file(TEXT_EMBEDDING_CACHE),
            **TEXT_EMBEDDING_METADATA,
            "primary_metric": "mean bidirectional normalized recall@k curve AUC over k/N",
            "checkpoint_alias": str(ckpt_dir / "best_val_normalized_recall_auc.pt"),
            "text_proj_init": "pretrained_infonce",
            "text_projection_trainable": True,
            "single_positive_policy": "one selected primary text per map",
        })
        write_json(complete_marker, {"status": "complete", "stage3_dir": str(stage3_dir)})
else:
    print("Normalized Stage 3 not requested")

if RUN_FRESH_STAGE3_PROJECTOR_CONTROL:
    print("Fresh-projector control is enabled, but not part of the primary six-run experiment. Add a separate control loop here if needed.")


# Stage 4: Run All Six Controlled Variants

Stage 4 must run for all six normalized Stage 3 variants, not only the Stage 3 winners. The goal is to compare whether domain-specific Stage 1B AE fine-tuning improves both Stage 3 retrieval and corrected Stage 4 text-to-brain generation.

Corrected 6a outputs are written under `corrected_stage4_normalized_specter/`.

All comparisons are within-domain only. Baseline and specialized runs for the same domain must use identical train, validation, and test splits, text IDs, SPECTER embeddings, batching, optimizer, schedule, random seed where practical, loss, early stopping, checkpoint-selection policy, and evaluation implementation.

## Selected AE Checkpoints

Use the explicit selected-checkpoint registry produced above. Do not infer checkpoints from filename labels, directory order, mtime, the most recent run, or generic aliases.

Required choices:

| Registry key | Checkpoint | Selection reason |
| --- | --- | --- |
| `mixed_stage1a` | `mixed_baseline_raw_mse/checkpoints/best_top1_dice.pt` | `held_out_multi_source_rank_1` |
| `mixed_to_pubmed_stage1b` | `pubmed/checkpoints/best_top1_dice.pt` | `held_out_domain_rank_1` |
| `mixed_to_nilearn_stage1b` | `nilearn/checkpoints/best_val_loss.pt` | `held_out_top5_dice_rank_1` |
| `mixed_to_neurovault_stage1b` | `neurovault/checkpoints/best_top5_dice.pt` | `held_out_top5_dice_rank_1` |

## Six Stage 4 Runs

| Stage 4 run | Domain | Type | AE registry key | Stage 3 source |
| --- | --- | --- | --- | --- |
| `mixed_stage1a_on_pubmed_stage4` | PubMed | baseline | `mixed_stage1a` | `mixed_stage1a_on_pubmed` |
| `mixed_to_pubmed_stage1b_on_pubmed_stage4` | PubMed | specialized | `mixed_to_pubmed_stage1b` | `mixed_to_pubmed_stage1b_on_pubmed` |
| `mixed_stage1a_on_nilearn_stage4` | Nilearn | baseline | `mixed_stage1a` | `mixed_stage1a_on_nilearn` |
| `mixed_to_nilearn_stage1b_on_nilearn_stage4` | Nilearn | specialized | `mixed_to_nilearn_stage1b` | `mixed_to_nilearn_stage1b_on_nilearn` |
| `mixed_stage1a_on_neurovault_stage4` | NeuroVault | baseline | `mixed_stage1a` | `mixed_stage1a_on_neurovault` |
| `mixed_to_neurovault_stage1b_on_neurovault_stage4` | NeuroVault | specialized | `mixed_to_neurovault_stage1b` | `mixed_to_neurovault_stage1b_on_neurovault` |

## Component Matching

Each Stage 4 run must load only matching upstream components:

1. Stage 3 text-side representation/projection from the matching Stage 3 run.
2. AE decoder from the exact AE checkpoint used to initialize that Stage 3 run.
3. Correct domain-specific train, validation, and held-out test splits.
4. A new Stage 4 text-to-latent projection head unique to that run.

Fail before training if a specialized Stage 3 checkpoint is paired with the mixed decoder, a mixed Stage 3 checkpoint is paired with a Stage 1B decoder, components are crossed across domains, a projection head is reused, or a run is evaluated on the wrong domain test split.

## Required Preflight And Provenance

Before each Stage 4 run, write `stage4_component_provenance.json` and `stage4_trainable_parameter_report.json`.

Validate and record:

* AE checkpoint path, filename, stage, training domain, selection reason, epoch, encoder checksum, decoder checksum.
* Stage 3 run/checkpoint path, epoch, selection metric, text projection checksum.
* SPECTER model/version.
* Train/validation/test split fingerprints.
* Decoder, Stage 3 text projection, and Stage 4 projection trainable/frozen status.
* Stage 3 domain equals Stage 4 domain.
* Stage 3 AE initialization path equals the AE checkpoint supplying the decoder.
* Decoder checksum matches the registered AE checkpoint.
* Baseline/specialized split fingerprints match within domain.
* No test example appears in train or validation data.

Dimensional preflight must verify:

```text
text batch -> SPECTER/text embedding -> Stage 3 text projection -> Stage 4 latent [batch, 384] -> decoder output [batch, 1, 36, 45, 38]
```

Primary trainability policy:

* AE decoder frozen.
* Stage 4 text-to-AE-latent projection trainable.
* Stage 3 CNN encoder not updated for generation.
* Stage 3 text projection frozen unless the previously validated Stage 4 recipe trained it; if so, use that same policy for all six runs and log it.
* SPECTER/base embeddings unchanged.

## Checkpointing, Evaluation, And Outputs

For each Stage 4 run, save:

* `best_val_loss.pt`
* `best_val_spatial_corr.pt`
* `best_val_top5_dice.pt`
* `best_val_foreground_mse.pt`
* `last.pt`
* training/validation histories and per-example held-out test generation metrics
* generated maps or compact prediction tensor plus manifest

Do not select final generation checkpoint by validation MSE alone. Prefer validation spatial correlation, top-5 Dice/overlap, foreground reconstruction quality, then validation MSE.

Test split is evaluation-only: no gradients, early stopping, checkpoint selection, or hyperparameter decisions.

## Comparison Files

Create within-domain Stage 4 comparisons:

* `pubmed_stage4_baseline_vs_specialized.csv`
* `nilearn_stage4_baseline_vs_specialized.csv`
* `neurovault_stage4_baseline_vs_specialized.csv`

Also create:

* `all_domain_stage4_comparison.csv`
* `ae_retrieval_generation_comparison.csv`

Primary conclusions must come from within-domain paired comparisons, not raw cross-domain ranking.

## Completion Rule

A Stage 4 run is complete only if saved outputs exist and validate: config, provenance, trainable-parameter report, checkpoint, training/validation histories, held-out generation metrics, nonzero predictions, per-example metrics, and generated-map manifest. The overall pipeline is complete only when all six Stage 4 variants are complete.

In [ ]:
def discover_stage3_checkpoint(domain: str, branch: str) -> Path:
    if RERUN_STAGE3:
        candidate = RUN_DIR / domain_dirs[domain] / branch / "stage3_normalized_specter/checkpoints/best_val_normalized_recall_auc.pt"
    else:
        if COMPLETED_STAGE3_RUN_ROOT is None:
            raise RuntimeError(
                "RERUN_STAGE3 is False, so Stage 4 needs explicit completed Stage 3 outputs. "
                "Set NEUROVLM_COMPLETED_STAGE3_RUN_ROOT to the run directory containing 01_pubmed/, 02_nilearn/, and 03_neurovault/."
            )
        candidate = COMPLETED_STAGE3_RUN_ROOT / domain_dirs[domain] / branch / "stage3_normalized_specter/checkpoints/best_val_normalized_recall_auc.pt"
    if not candidate.exists():
        raise FileNotFoundError(f"Missing matching Stage 3 checkpoint: {candidate}")
    return candidate


def discover_stage4_generation_cache() -> Path:
    if NORMALIZED_SPECTER_CACHE is None or not Path(NORMALIZED_SPECTER_CACHE).exists():
        raise FileNotFoundError(f"Normalized SPECTER2 cache does not exist: {NORMALIZED_SPECTER_CACHE}")
    return Path(NORMALIZED_SPECTER_CACHE)



STAGE4_GENERATION_TEXT_CACHE = discover_stage4_generation_cache() if RUN_STAGE4_TEXT_TO_BRAIN else None
if STAGE4_GENERATION_TEXT_CACHE:
    print("Corrected Stage 4 generation cache:", STAGE4_GENERATION_TEXT_CACHE)

if RUN_STAGE4_TEXT_TO_BRAIN:
    if not STAGE2_STAGE3_STAGE4_INPUT_MANIFEST or not Path(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST).exists():
        raise RuntimeError("Stage 4 requires the validated Stage 1 selected-checkpoint manifest")
    downstream_manifest = json.loads(Path(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST).read_text())
    domain_dirs = {"pubmed": "01_pubmed", "nilearn": "02_nilearn", "neurovault": "03_neurovault"}
    specialized_dirs = {
        "pubmed": "specialized_mixed_to_pubmed",
        "nilearn": "specialized_mixed_to_nilearn",
        "neurovault": "specialized_mixed_to_neurovault",
    }
    stage4_epochs = int(os.environ.get("NEUROVLM_STAGE4_EPOCHS", "200"))
    stage4_patience = int(os.environ.get("NEUROVLM_STAGE4_EARLY_STOPPING_PATIENCE", "25"))
    stage4_val_interval = int(os.environ.get("NEUROVLM_STAGE4_VAL_INTERVAL", "1"))
    generation_auc_val_interval = int(os.environ.get("NEUROVLM_GENERATION_AUC_VAL_INTERVAL", "5"))
    stage4_summaries = []
    for run in downstream_manifest["six_stage2_stage3_stage4_runs"]:
        domain = run["domain"]
        branch = "baseline_mixed_stage1a" if run["type"] == "baseline" else specialized_dirs[domain]
        branch_dir = RUN_DIR / domain_dirs[domain] / branch
        stage3_ckpt = discover_stage3_checkpoint(domain, branch)
        stage3_dir = stage3_ckpt.parents[1]
        stage4_dir = branch_dir / "corrected_stage4_normalized_specter"
        ckpt_dir = stage4_dir / "checkpoints"
        config_dir = stage4_dir / "config"
        for rel in ["config", "provenance", "checkpoints", "training_metrics", "validation_metrics", "test_metrics", "per_example", "recall_curves", "generated_maps", "plots"]:
            (stage4_dir / rel).mkdir(parents=True, exist_ok=True)
        ae_entry = downstream_manifest["selected_ae_checkpoints"][run["ae_registry_key"]]
        provenance = {
            "stage4_run": f"{run['run']}_normalized_stage4",
            "domain": domain,
            "type": run["type"],
            "architecture": "generative_text_to_ae_latent",
            "ae_registry_key": run["ae_registry_key"],
            "ae_checkpoint_path": ae_entry["path"],
            "ae_checkpoint_name": ae_entry["checkpoint_name"],
            "ae_stage": ae_entry["stage"],
            "ae_training_domain": ae_entry["training_domain"],
            "ae_selection_reason": ae_entry["selection_reason"],
            "stage3_dir": str(stage3_dir),
            "stage3_checkpoint": str(stage3_ckpt),
            "stage3_usage": "semantic_validation_and_test_evaluation_only",
            "train_jsonl": TRAIN_JSONL,
            "val_jsonl": VAL_JSONL,
            "test_jsonl": TEST_JSONL,
            "stage4_text_embedding_cache": str(STAGE4_GENERATION_TEXT_CACHE),
            "stage4_text_embedding_cache_sha256": sha256_file(STAGE4_GENERATION_TEXT_CACHE),
            **TEXT_EMBEDDING_METADATA,
            "single_positive_policy": "one selected primary text per map",
        }
        write_json(stage4_dir / "provenance/stage4_component_provenance.json", provenance)
        write_json(stage4_dir / "stage4_trainable_parameter_report.json", {
            "frozen": ["autoencoder_encoder", "autoencoder_decoder", "stage3_contrastive_projector", "stage3_brain_encoder"],
            "trainable": ["generative_text_to_ae_latent"],
            "forbidden": ["stage3_contrastive_projector_weight_loading", "stage3_contrastive_embedding_decoder_input"],
            "note": "Corrected Stage 4 learns a fresh 768->512->384 projector into the raw frozen AE latent space.",
        })
        cfg = {
            "train_jsonl": TRAIN_JSONL,
            "val_jsonl": VAL_JSONL,
            "test_jsonl": TEST_JSONL,
            "eval_jsonls": {},
            "text_embedding_cache": str(STAGE4_GENERATION_TEXT_CACHE),
            **TEXT_EMBEDDING_METADATA,
            "autoencoder_checkpoint": ae_entry["path"],
            "stage3_contrastive_checkpoint": str(stage3_ckpt),
            "domain": domain,
            "output_dir": str(stage4_dir),
            "checkpoint_dir": str(ckpt_dir),
            "device": "auto",
            "seed": 42,
            "target_shape": [36, 45, 38],
            "batch_size": int(os.environ.get("NEUROVLM_STAGE4_BATCH_SIZE", "1024")),
            "preflight_batch_size": True,
            "batch_candidates": [4096, 3072, 2048, 1536, 1024, 768, 512, 384, 256, 192, 128, 96, 64],
            "runtime_batch_fallback": True,
            "num_workers": NUM_WORKERS,
            "pin_memory": True,
            "persistent_workers": NUM_WORKERS > 0,
            "prefetch_factor": PREFETCH_FACTOR,
            "epochs": stage4_epochs,
            "early_stopping": True,
            "early_stopping_metric": "val_generation_normalized_auc",
            "early_stopping_mode": "max",
            "early_stopping_patience": stage4_patience,
            "early_stopping_min_delta": 0.0,
            "lr": float(os.environ.get("NEUROVLM_STAGE4_LR", "0.00005")),
            "weight_decay": float(os.environ.get("NEUROVLM_STAGE4_WEIGHT_DECAY", "0.0001")),
            "positive_texts_per_map": 1,
            "text_projection_init": "random",
            "prediction_activation": "none",
            "amp": True,
            "cudnn_benchmark": True,
            "val_interval": stage4_val_interval,
            "generation_auc_val_interval": generation_auc_val_interval,
            "generation_auc_batch_size": int(os.environ.get("NEUROVLM_GENERATION_AUC_BATCH_SIZE", "512")),
            "compute_train_metrics": True,
            "train_metric_batches": 8,
            "val_metric_batches": 16,
            "metrics_device": METRICS_DEVICE,
            "include_voxel_auroc": False,
            "eval_num_workers": EVAL_NUM_WORKERS,
            "model": {
                "latent_dim": 384,
                "base_channels": 64,
                "num_blocks": 4,
                "encoder_arch": "plain",
                "dropout": 0.1,
                "norm": "group",
                "pooling": "max",
                "blocks_per_stage": 2,
                "use_dilation": False,
                "multi_scale": False,
                "global_context": "none",
            },
            "generative_text_to_ae_latent": {"name": "generative_text_to_ae_latent", "in_dim": 768, "hidden_dim": 512, "latent_dim": 384},
            "weighted_recon": {"type": "mse", "alpha": 0.0, "gamma": 1.0, "normalize_target": True},
            "loss_name": "latent_mse_plus_reconstruction_mse",
            "loss": {"lambda_recon": 1.0, "lambda_latent": 1.0, "lambda_dice": 0.0, "lambda_topk": 0.0, "lambda_corr": 0.0},
            "optional_loss_ablations_supported_not_run_by_default": [
                "latent_mse_only",
                "latent_mse_plus_reconstruction_mse",
                "latent_mse_plus_cosine_plus_reconstruction_mse",
            ],
        }
        config_path = config_dir / "text_to_brain_config.json"
        write_json(config_path, cfg)
        cmd = [sys.executable, "-m", "atlas_free_cnn.training.train_text_to_brain", "--config", str(config_path)]
        env = os.environ.copy()
        env["PYTHONPATH"] = os.pathsep.join([
            str(REPO_DIR / "experiments" / "3dcnn"),
            str(REPO_DIR / "src"),
            str(REPO_DIR),
            env.get("PYTHONPATH", ""),
        ])
        print("Launching corrected Stage 4", f"{run['run']}_normalized_stage4")
        print(" ".join(cmd))
        env["PYTHONUNBUFFERED"] = "1"
        _proc4 = subprocess.Popen(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1, env=env)
        for _line in _proc4.stdout:
            print(_line, end='', flush=True)
        _proc4.wait()
        if _proc4.returncode != 0:
            raise RuntimeError(f"Corrected Stage 4 run failed for {run['run']} with exit code {_proc4.returncode}")
        stage4_summaries.append({"run": f"{run['run']}_normalized_stage4", "stage4_dir": str(stage4_dir), "config": str(config_path), "primary_checkpoint": str(ckpt_dir / "best_val_generation_normalized_auc.pt")})
    write_json(Path(paths["stage4"]) / "corrected_stage4_run_manifest.json", stage4_summaries)
else:
    print("Corrected Stage 4 not requested")

if "stage5" not in paths:
    paths["stage5"] = str(RUN_DIR / "04_stage5_generation_eval")
    Path(paths["stage5"]).mkdir(parents=True, exist_ok=True)

if RUN_STAGE5_GENERATION_EVAL:
    print("Corrected Stage 4 semantic/spatial test evaluation is produced by the trainer and notebook 7 diagnostics.")
else:
    print("Stage 5 not requested")


## Final Status and Comparison Files

In [ ]:
stage_status = write_status_report(
    RUN_DIR,
    {
        "stage1_selection_integration": stage1_selection_status,
        "stage3_normalized_specter": RUN_STAGE3_CONTRASTIVE,
        "corrected_stage4_normalized_specter": RUN_STAGE4_TEXT_TO_BRAIN,
        "stage5": RUN_STAGE5_GENERATION_EVAL,
    },
    layout="normalized_specter"
)

def read_json_if_exists(path: Path):
    try:
        return json.loads(path.read_text()) if path.exists() else {}
    except Exception:
        return {}

if STAGE1_SELECTION_INTEGRATION_DIR:
    assignment_csv = STAGE1_SELECTION_INTEGRATION_DIR / "03_downstream_usage/six_run_ae_assignment.csv"
    final_assignment = Path(paths["final"]) / "six_run_ae_assignment.csv"
    if assignment_csv.exists():
        shutil.copy2(assignment_csv, final_assignment)
    write_table(Path(paths["final"]) / "final_summary_table.csv", [{"stage": s["stage"], "status": s["status"], "expected_runs": s.get("expected_runs", ""), "completed_runs": s.get("completed_runs", "")} for s in stage_status])
    write_json(Path(paths["final"]) / "stage1_selection_integration_pointer.json", {
        "integration_dir": str(STAGE1_SELECTION_INTEGRATION_DIR),
        "downstream_manifest": str(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST),
    })
else:
    write_table(Path(paths["final"]) / "final_summary_table.csv", [{"stage": s["stage"], "status": s["status"], "expected_runs": s.get("expected_runs", ""), "completed_runs": s.get("completed_runs", "")} for s in stage_status])

audit_rows = []
try:
    manifest_for_audit = json.loads(Path(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST).read_text()) if STAGE2_STAGE3_STAGE4_INPUT_MANIFEST else {"six_stage2_stage3_stage4_runs": []}
    pair_fingerprints = {}
    for run in manifest_for_audit["six_stage2_stage3_stage4_runs"]:
        domain = run["domain"]
        branch = "baseline_mixed_stage1a" if run["type"] == "baseline" else specialized_dirs[domain]
        branch_dir = RUN_DIR / domain_dirs[domain] / branch
        stage3_arch = read_json_if_exists(branch_dir / f"{NORMALIZED_STAGE3_DIRNAME}/architecture_compatibility_report.json")
        stage3_train = read_json_if_exists(branch_dir / f"{NORMALIZED_STAGE3_DIRNAME}/stage3_trainability_report.json")
        stage3_data = read_json_if_exists(branch_dir / f"{NORMALIZED_STAGE3_DIRNAME}/domain_dataset_report.json")
        stage4_arch = read_json_if_exists(branch_dir / f"{CORRECTED_STAGE4_DIRNAME}/stage4_architecture_compatibility_report.json")
        selected_arch = stage3_arch.get("selected_ae_checkpoint_architecture", {})
        instantiated = stage3_arch.get("instantiated_stage3_architecture", {})
        fps = stage3_data.get("split_fingerprints", {})
        stage4_domain = stage4_arch.get("domain_filter_report", {})
        stage4_fps = {name: stage4_domain.get(name, {}).get("fingerprint", "") for name in ["train", "val", "test"]}
        stage4_domain_filter_passed = bool(stage4_domain) and all(stage4_fps.get(name) == fps.get(name) for name in ["train", "val", "test"])
        pair_fingerprints.setdefault(domain, []).append(fps)
        audit_rows.append({
            "run_name": run["run"],
            "domain": domain,
            "baseline_or_specialized": run["type"],
            "AE checkpoint": run.get("ae_checkpoint_path", ""),
            "AE base channels": selected_arch.get("base_channels", ""),
            "Stage 3 base channels": instantiated.get("base_channels", ""),
            "number of blocks": instantiated.get("num_blocks", ""),
            "output dimension": instantiated.get("out_dim", ""),
            "norm": instantiated.get("norm", ""),
            "pooling": instantiated.get("pooling", ""),
            "dropout": instantiated.get("dropout", ""),
            "encoder architecture": instantiated.get("encoder_arch", ""),
            "strict encoder load passed": stage3_arch.get("strict_load_success", ""),
            "CNN encoder trainable": stage3_train.get("cnn_encoder_trainable", ""),
            "text projection pretrained": stage3_train.get("text_projection_pretrained", ""),
            "text projection trainable": stage3_train.get("text_projection_trainable", ""),
            "symmetric InfoNCE confirmed": stage3_train.get("loss", "") == "symmetric InfoNCE",
            "train source counts": json.dumps(stage3_data.get("source_value_counts", {}).get("train", {}), sort_keys=True),
            "validation source counts": json.dumps(stage3_data.get("source_value_counts", {}).get("val", {}), sort_keys=True),
            "test source counts": json.dumps(stage3_data.get("source_value_counts", {}).get("test", {}), sort_keys=True),
            "train fingerprint": fps.get("train", ""),
            "validation fingerprint": fps.get("val", ""),
            "test fingerprint": fps.get("test", ""),
            "matching-pair fingerprints equal": "pending",
            "Stage 4 decoder strict load passed": stage4_arch.get("strict_autoencoder_load", "") == "passed",
            "Stage 4 domain filter passed": stage4_domain_filter_passed,
            "warnings": json.dumps({"stage3_arch": stage3_arch.get("unexpected_differences", {}), "stage3_train": stage3_train.get("failures", [])}, sort_keys=True),
            "status": "passed" if stage3_arch.get("final_compatibility_status") == "passed" and stage3_train.get("status") == "passed" else "incomplete_or_failed",
        })
    for row in audit_rows:
        fps_group = pair_fingerprints.get(row["domain"], [])
        row["matching-pair fingerprints equal"] = len(fps_group) == 2 and fps_group[0] == fps_group[1]
    if audit_rows:
        write_table(Path(paths["final"]) / "architecture_and_data_audit.csv", audit_rows)
except Exception as exc:
    print("WARNING: could not write architecture_and_data_audit.csv:", exc)

for s in stage_status:
    print(f"{s['stage']}: {s['status']} ({s.get('completed_runs', '')}/{s.get('expected_runs', '')})")
print("Final comparison:", Path(paths["final"]))

summary_root = RUN_DIR / "03_all_domain_summary"
cache_audit_dir = RUN_DIR / "00_normalized_text_cache_audit"
stage3_summary_dir = RUN_DIR / "01_stage3_normalized"
stage4_summary_dir = RUN_DIR / "02_corrected_stage4_normalized"
for d in [summary_root, cache_audit_dir, stage3_summary_dir, stage4_summary_dir]:
    d.mkdir(parents=True, exist_ok=True)
for src in [NORMALIZED_PATHS.get("metadata"), NORMALIZED_PATHS.get("validation"), NORMALIZED_PATHS.get("index")]:
    try:
        if src and Path(src).exists():
            shutil.copy2(src, cache_audit_dir / Path(src).name)
    except Exception as exc:
        print("Cache audit copy skipped:", exc)


def first_metric(payloads, names):
    for payload in payloads:
        if isinstance(payload, list):
            for row in payload:
                if isinstance(row, dict) and row.get("source") == "all":
                    value = first_metric([row], names)
                    if value != "":
                        return value
            continue
        if not isinstance(payload, dict):
            continue
        test = payload.get("test", payload)
        for name in names:
            value = test.get(name)
            if value not in {None, ""}:
                return value
    return ""


def to_float(value):
    try:
        return float(value)
    except Exception:
        return None


def delta(new_value, old_value):
    new_f = to_float(new_value)
    old_f = to_float(old_value)
    return "" if new_f is None or old_f is None else new_f - old_f


def load_stage3_metric_bundle(stage3_dir: Path) -> dict:
    payloads = [
        read_json_if_exists(stage3_dir / "eval_results.json"),
        read_json_if_exists(stage3_dir / "test_metrics.json"),
        read_json_if_exists(stage3_dir / "metrics/test_metrics.json"),
        read_json_if_exists(stage3_dir / "comparison_row.json"),
    ]
    t2b = first_metric(payloads, ["t2i_normalized_k_recall_curve_auc", "text_to_brain_normalized_auc", "text_to_image_auc", "t2i_auc"])
    b2t = first_metric(payloads, ["i2t_normalized_k_recall_curve_auc", "brain_to_text_normalized_auc", "image_to_text_auc", "i2t_auc"])
    mean_auc = first_metric(payloads, ["paper_recall_curve_auc", "mean_bidirectional_normalized_auc", "mean_normalized_k_recall_curve_auc", "normalized_k_recall_curve_auc", "full_recall_curve_auc"])
    if mean_auc == "" and to_float(t2b) is not None and to_float(b2t) is not None:
        mean_auc = (float(t2b) + float(b2t)) / 2.0
    return {"mean_auc": mean_auc, "text_to_brain_auc": t2b, "brain_to_text_auc": b2t}


def load_stage4_all_row(stage4_dir: Path) -> dict:
    payload = read_json_if_exists(stage4_dir / "generation_eval_metrics.json")
    if isinstance(payload, list):
        return next((r for r in payload if isinstance(r, dict) and r.get("source") == "all"), payload[0] if payload and isinstance(payload[0], dict) else {})
    return payload if isinstance(payload, dict) else {}

stage3_rows = []
stage4_rows = []
stage3_compare_rows = []
stage4_compare_rows = []
combined_rows = []
try:
    manifest = json.loads(Path(STAGE2_STAGE3_STAGE4_INPUT_MANIFEST).read_text()) if STAGE2_STAGE3_STAGE4_INPUT_MANIFEST else {"six_stage2_stage3_stage4_runs": []}
    for run in manifest["six_stage2_stage3_stage4_runs"]:
        domain = run["domain"]
        branch_kind = run["type"]
        branch = "baseline_mixed_stage1a" if branch_kind == "baseline" else specialized_dirs[domain]
        branch_dir = RUN_DIR / domain_dirs[domain] / branch
        base_run = run["run"]
        stage3_dir = branch_dir / NORMALIZED_STAGE3_DIRNAME
        legacy_stage3_dir = COMPLETED_STAGE3_RUN_ROOT / domain_dirs[domain] / branch / "stage3"
        normalized_stage3 = load_stage3_metric_bundle(stage3_dir)
        legacy_stage3 = load_stage3_metric_bundle(legacy_stage3_dir)
        stage3_arch = read_json_if_exists(stage3_dir / "architecture_compatibility_report.json")
        stage3_data = read_json_if_exists(stage3_dir / "domain_dataset_report.json")
        fps = stage3_data.get("split_fingerprints", {})
        split_fp = fps.get("test", SPLIT_FINGERPRINTS.get("test", {}).get("fingerprint", ""))
        stage3_row = {
            "run": f"{base_run}_normalized_specter",
            "domain": domain,
            "branch": branch,
            "branch_kind": branch_kind,
            "normalized_mean_bidirectional_auc": normalized_stage3["mean_auc"],
            "normalized_text_to_brain_auc": normalized_stage3["text_to_brain_auc"],
            "normalized_brain_to_text_auc": normalized_stage3["brain_to_text_auc"],
            "checkpoint": str(stage3_dir / "checkpoints" / NORMALIZED_STAGE3_CHECKPOINT),
            "normalized_cache_sha256": TEXT_EMBEDDING_METADATA["text_embedding_cache_checksum"],
            **TEXT_EMBEDDING_METADATA,
            "split_fingerprint": split_fp,
            "architecture_match": stage3_arch.get("final_compatibility_status", "") == "passed" if stage3_arch else "",
            "text_preprocessing_change_only": True,
        }
        stage3_rows.append(stage3_row)
        stage3_compare_rows.append({
            "domain": domain,
            "branch": branch,
            "legacy_mean_bidirectional_auc": legacy_stage3["mean_auc"],
            "normalized_mean_bidirectional_auc": normalized_stage3["mean_auc"],
            "delta_normalized_minus_legacy": delta(normalized_stage3["mean_auc"], legacy_stage3["mean_auc"]),
            "legacy_text_to_brain_auc": legacy_stage3["text_to_brain_auc"],
            "normalized_text_to_brain_auc": normalized_stage3["text_to_brain_auc"],
            "legacy_brain_to_text_auc": legacy_stage3["brain_to_text_auc"],
            "normalized_brain_to_text_auc": normalized_stage3["brain_to_text_auc"],
            "split_fingerprint": split_fp,
            "architecture_match": stage3_row["architecture_match"],
            "text_preprocessing_change_only": True,
        })

        stage4_dir = branch_dir / CORRECTED_STAGE4_DIRNAME
        all_row = load_stage4_all_row(stage4_dir)
        corrected_auc = all_row.get("generation_mean_normalized_auc", all_row.get("generation_clamped_strict_map_mean_normalized_auc", ""))
        legacy_auc = ""
        corrected_checkpoint = stage4_dir / "checkpoints" / CORRECTED_STAGE4_CHECKPOINT
        stage4_row = {
            "run": f"{base_run}_corrected_normalized_stage4",
            "domain": domain,
            "branch": branch,
            "branch_kind": branch_kind,
            "corrected_generation_mean_auc": corrected_auc,
            "text_to_generated_brain_auc": all_row.get("generation_text_to_brain_normalized_auc", ""),
            "generated_brain_to_text_auc": all_row.get("generation_brain_to_text_normalized_auc", ""),
            "raw_strict_map_auc": all_row.get("generation_raw_strict_map_mean_normalized_auc", ""),
            "clamped_strict_map_auc": all_row.get("generation_clamped_strict_map_mean_normalized_auc", ""),
            "same_text_group_auc": all_row.get("generation_clamped_same_text_group_mean_normalized_auc", ""),
            "publication_group_auc": all_row.get("generation_clamped_publication_group_mean_normalized_auc", ""),
            "matched_contrastive_cosine": all_row.get("generation_matched_contrastive_cosine", all_row.get("generation_clamped_matched_contrastive_cosine", "")),
            "shuffled_null_contrastive_cosine": all_row.get("generation_shuffled_contrastive_cosine", all_row.get("generation_clamped_shuffled_contrastive_cosine", "")),
            "spatial_corr": all_row.get("spatial_corr", ""),
            "top1_dice": all_row.get("top1_dice", ""),
            "top5_dice": all_row.get("top5_dice", ""),
            "top10_dice": all_row.get("top10_dice", ""),
            "mse": all_row.get("mse", ""),
            "mae": all_row.get("mae", ""),
            "foreground_mse": all_row.get("foreground_mse", ""),
            "pred_nonzero_frac": all_row.get("pred_nonzero_fraction", all_row.get("pred_nonzero_frac", "")),
            "target_nonzero_frac": all_row.get("target_nonzero_fraction", all_row.get("target_nonzero_frac", "")),
            "primary_checkpoint": str(corrected_checkpoint),
        }
        stage4_rows.append(stage4_row)
        stage4_compare_rows.append({
            "domain": domain,
            "branch": branch,
            "legacy_generation_mean_auc": legacy_auc,
            "corrected_generation_mean_auc": corrected_auc,
            "delta_corrected_minus_legacy": delta(corrected_auc, legacy_auc),
            "corrected_raw_strict_map_auc": stage4_row["raw_strict_map_auc"],
            "corrected_clamped_strict_map_auc": stage4_row["clamped_strict_map_auc"],
            "corrected_same_text_group_auc": stage4_row["same_text_group_auc"],
            "corrected_publication_group_auc": stage4_row["publication_group_auc"],
            "corrected_spatial_corr": stage4_row["spatial_corr"],
            "corrected_top5_dice": stage4_row["top5_dice"],
            "corrected_pred_nonzero_frac": stage4_row["pred_nonzero_frac"],
            "corrected_target_nonzero_frac": stage4_row["target_nonzero_frac"],
            "legacy_checkpoint_name_or_source": "notebook6_saved_legacy_stage4_auc_constant",
            "corrected_checkpoint_name": CORRECTED_STAGE4_CHECKPOINT,
        })
        combined_rows.append({"section": "stage3", **stage3_row})
        combined_rows.append({"section": "stage4", **stage4_row})
except Exception as exc:
    print("WARNING: summary packaging incomplete:", exc)

write_table(summary_root / "stage3_normalized_all_runs.csv", stage3_rows)
write_table(summary_root / "stage4_corrected_normalized_all_runs.csv", stage4_rows)
write_table(summary_root / "normalized_vs_legacy_stage3.csv", stage3_compare_rows)
write_table(summary_root / "normalized_corrected_vs_legacy_stage4.csv", stage4_compare_rows)
write_table(summary_root / "stage1_stage3_stage4_normalized_summary.csv", combined_rows)
write_table(stage3_summary_dir / "normalized_vs_legacy_stage3.csv", stage3_compare_rows)
write_table(stage4_summary_dir / "normalized_corrected_vs_legacy_stage4.csv", stage4_compare_rows)


def mean_delta(rows, key):
    vals = [to_float(r.get(key)) for r in rows]
    vals = [v for v in vals if v is not None]
    return None if not vals else sum(vals) / len(vals)


def answer_from_delta(value, positive_text, negative_text, missing_text="insufficient completed metrics"):
    if value is None:
        return missing_text
    if value > 0:
        return f"yes, mean delta {value:.6f} ({positive_text})"
    if value < 0:
        return f"no, mean delta {value:.6f} ({negative_text})"
    return "no material change, mean delta 0.000000"

stage3_delta = mean_delta(stage3_compare_rows, "delta_normalized_minus_legacy")
stage4_delta = mean_delta(stage4_compare_rows, "delta_corrected_minus_legacy")
raw_clamped_changes = [
    abs(to_float(r.get("corrected_raw_strict_map_auc")) - to_float(r.get("corrected_clamped_strict_map_auc")))
    for r in stage4_compare_rows
    if to_float(r.get("corrected_raw_strict_map_auc")) is not None and to_float(r.get("corrected_clamped_strict_map_auc")) is not None
]
raw_clamped_answer = "insufficient corrected Stage 4 metrics"
if raw_clamped_changes:
    raw_clamped_answer = f"mean absolute raw-vs-clamped strict AUC difference {sum(raw_clamped_changes) / len(raw_clamped_changes):.6f}; inspect branches where this is nonzero"

neurovault_rows = [r for r in stage4_compare_rows if r.get("domain") == "neurovault"]
duplicate_answer = "insufficient NeuroVault corrected Stage 4 metrics"
if neurovault_rows:
    duplicate_answer = "; ".join(
        f"{r['branch']}: strict={r.get('corrected_clamped_strict_map_auc', '')}, same_text={r.get('corrected_same_text_group_auc', '')}, publication={r.get('corrected_publication_group_auc', '')}"
        for r in neurovault_rows
    )

best_by_domain = []
for domain in ["pubmed", "nilearn", "neurovault"]:
    candidates = [r for r in stage4_rows if r.get("domain") == domain and to_float(r.get("corrected_generation_mean_auc")) is not None]
    metric_name = "corrected Stage 4 generation AUC"
    if not candidates:
        candidates = [r for r in stage3_rows if r.get("domain") == domain and to_float(r.get("normalized_mean_bidirectional_auc")) is not None]
        metric_name = "normalized Stage 3 AUC"
    if candidates:
        best = max(candidates, key=lambda r: to_float(r.get("corrected_generation_mean_auc", r.get("normalized_mean_bidirectional_auc"))) or -1e9)
        score = best.get("corrected_generation_mean_auc", best.get("normalized_mean_bidirectional_auc", ""))
        best_by_domain.append(f"{domain}: {best['branch']} by {metric_name} ({score})")
    else:
        best_by_domain.append(f"{domain}: unavailable until metrics are exported")

recommend_stage3 = "yes" if stage3_delta is not None and stage3_delta > 0 else "not yet" if stage3_delta is not None else "unknown"
recommend_stage4 = "yes" if stage4_delta is not None and stage4_delta > 0 else "not yet" if stage4_delta is not None else "unknown"

README = f"""# 6a Normalized/Corrected Summary

1. Did unit-normalized SPECTER2 improve Stage 3 normalized recall AUC? {answer_from_delta(stage3_delta, 'normalized SPECTER2 improved over legacy/raw Stage 3', 'normalized SPECTER2 trailed legacy/raw Stage 3')}.
2. Did normalized Stage 3 beat the saved legacy/raw Stage 3 baseline? {answer_from_delta(stage3_delta, 'paired normalized rows beat the saved legacy rows on average', 'paired normalized rows did not beat the saved legacy rows on average')}.
3. Did corrected independent Stage 4 improve generation semantic AUC? {answer_from_delta(stage4_delta, 'corrected Stage 4 improved over saved legacy Stage 4 constants', 'corrected Stage 4 trailed saved legacy Stage 4 constants')}.
4. Did raw-vs-clamped semantic AUC change the interpretation? {raw_clamped_answer}.
5. Did duplicate-aware NeuroVault evaluation change the interpretation? {duplicate_answer}.
6. Which AE initialization is best per domain? {'; '.join(best_by_domain)}.
7. Should normalized SPECTER2 become the default for Stage 3 and/or Stage 4? Stage 3: {recommend_stage3}; Stage 4: {recommend_stage4}. Use the paired CSVs below as the decision record.

Primary files:
- `stage3_normalized_all_runs.csv`
- `stage4_corrected_normalized_all_runs.csv`
- `normalized_vs_legacy_stage3.csv`
- `normalized_corrected_vs_legacy_stage4.csv`
- `stage1_stage3_stage4_normalized_summary.csv`

Normalized cache: `{NORMALIZED_SPECTER_CACHE}`
Cache SHA-256: `{TEXT_EMBEDDING_METADATA['text_embedding_cache_checksum']}`
Preprocessing: `{TEXT_EMBEDDING_PREPROCESSING}`
Convention: `{TEXT_EMBEDDING_CONVENTION}`
"""
(summary_root / "README_WHAT_TO_LOOK_AT.md").write_text(README)
print("6a packaged outputs:", RUN_DIR)
print("Summary:", summary_root)
